In [1]:
library(parallel)
library(GenomicFeatures)
set.seed(1234)
library(repr)
library(motifmatchr)
library(fastmatch)
library(qlcMatrix)

library(ensembldb)
library(EnsDb.Hsapiens.v86)
library(AnnotationFilter)

library(Signac)
library(Seurat)
library(JASPAR2024)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg38)
library(patchwork)
library(ggplot2)
library(Matrix)
library(zoo)
library(tidyr)

source("../multiome_methods/function_calls.r")
source("../multiome_methods/peak_gene_relations.r")
source("../multiome_methods/binding_site_identification.r")
source("../multiome_methods/TF_activity.r")
source("../multiome_methods/signac_utils.r")

library(dplyr)

library(ggplot2)
library(hexbin)
library(RColorBrewer)
library(cowplot)
library(gridExtra)
library(ggExtra)

library(pheatmap)
library(purrr)
library(data.table)
library(Rsamtools)
library(stringi)


Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The follo

In [2]:
object_w_net <- readRDS("object_w_net_smooth_muscle_contraction.rds")

In [4]:
str(object_w_net@misc, max.level = 1)

List of 12
 $ markers.annotated        :'data.frame':	4000 obs. of  8 variables:
 $ peak_stats               :'data.frame':	134 obs. of  43 variables:
 $ peak_stats.filtered      :'data.frame':	17 obs. of  51 variables:
 $ motifs.inProximalPeaks   :List of 3
 $ motif_enrichment         :'data.frame':	2160 obs. of  16 variables:
 $ motifs.inDistalPeaks     :List of 1
 $ filtered.motif_enrichment:'data.frame':	888 obs. of  16 variables:
 $ motif_stats              :'data.frame':	888 obs. of  26 variables:
 $ motif2TF                 :List of 720
 $ TF2motif                 :List of 668
 $ seedNetwork              :List of 1
 $ context_subNetwork       :List of 1


In [8]:
colnames(object_w_net@misc$peak_stats.filtered)

[1] "cluster"                                         
 [2] "peak"                                            
 [3] "gene"                                            
 [4] "annotation"                                      
 [5] "regulatorType"                                   
 [6] "signac.scores"                                   
 [7] "signac.zscores"                                  
 [8] "signac.pvalues"                                  
 [9] "t-stat_highly.acc"                               
[10] "p.value-t.test_highly.acc"                       
[11] "p.value-t.test_highly.acc_Bonf"                  
[12] "p.value-t.test_highly.acc_BH"                    
[13] "expr_not_0"                                      
[14] "acc_not_0"                                       
[15] "expr_not_0.given_acc"                            
[16] "expr_and_acc_not_0"                              
[17] "FC.expr_given_acc"                               
[18] "expr_not_0.bg"                                   
[19] "acc_not_0.bg"                                    
[20] "expr_not_0.given_acc.bg"                         
[21] "expr_and_acc_not_0.bg"                           
[22] "FC.expr_given_acc.bg"                            
[23] "expr_not_0.bg_other_peaks.same_cluster"          
[24] "acc_not_0.bg_other_peaks.same_cluster"           
[25] "expr_not_0.given_acc.bg_other_peaks.same_cluster"
[26] "expr_and_acc_not_0.bg_other_peaks.same_cluster"  
[27] "FC.expr_given_acc.bg_other_peaks.same_cluster"   
[28] "n.bg_other_peaks.same_cluster"                   
[29] "expr_not_0.all"                                  
[30] "acc_not_0.all"                                   
[31] "expr_not_0.given_acc.all"                        
[32] "expr_and_acc_not_0.all"                          
[33] "FC.expr_given_acc.all"                           
[34] "expr_not_0.bg_other_peaks.all"                   
[35] "acc_not_0.bg_other_peaks.all"                    
[36] "expr_not_0.given_acc.bg_other_peaks.all"         
[37] "expr_and_acc_not_0.bg_other_peaks.all"           
[38] "FC.expr_given_acc.bg_other_peaks.all"            
[39] "n.bg_other_peaks.all"                            
[40] "high_cor_distal"                                 
[41] "high_cor_distal_zScore"                          
[42] "promotersLinkedToSeed"                           
[43] "distalPeaksLinkedToSeed"                         
[44] "acc_cells_cluster"                               
[45] "delta_expr_given_acc.same_peak_bg"               
[46] "delta_expr_given_acc.other_peaks.same_cluster"   
[47] "delta_expr_given_acc.other_peaks.all"            
[48] "pass_cluster_specific"                           
[49] "pass_global"                                     
[50] "pass_any"                                        
[51] "pass_type"

## explainations peak-stats 
#### Identity columns 
- 'cluster'
- 'peak'
- 'gene'
- 'annotation': seed gene, or module gene (from prior net), or marker gene
- 'regulatorType': proximal: 100-2000 upstream TSS, else distal

#### LinkPeaks stats: across all clusters
- 'signac.scores': The raw Signac link score from Links(object)$score
- 'signac.zscores': The Signac link z-score from Links(object)$zscore
- 'signac.pvalues': The Signac link p-value from Links(object)$pvalue

#### Cluster-specific peak accessibility test columns: conduct_stat_test() using test='t-test', test_activation=TRUE (is this peak more accessible in the target cluster than outside it?)
foreground: cells of cluster,  background: all other cells, alternative: greater
- t-stat_highly.acc: The one-sided t-test statistic for target-cluster accessibility > background accessibility
- p.value-t.test_highly.acc: The raw p-value from that one-sided t-test
- p.value-t.test_highly.acc_Bonf: Bonferroni-adjusted p-value across all unique (cluster, peak) tests
- p.value-t.test_highly.acc_BH: Benjamini–Hochberg adjusted p-value across all unique (cluster, peak) tests

#### Probability / zero-expression columns: zero_expression_stats() and calc_prob_stats(gene_expr, peak_acc)
(n_cells = number of cells in cluster
n_expr = # cells with gene_expr != 0
n_acc = # cells with peak_acc != 0
n_joint = # cells with gene_expr != 0 AND peak_acc != 0)

- expr_not_0 = n_expr / n_cells: Fraction of cells in the target cluster where the gene is expressed
- acc_not_0 = n_acc / n_cells: Fraction of cells in the target cluster where this peak is accessible
- expr_not_0.given_acc = n_joint / n_acc: Conditional probability that the gene is expressed among cells where this peak is accessible, within the target cluster: P(expr != 0 | acc != 0) in the cluster
- expr_and_acc_not_0 = n_joint / n_cells: Joint fraction of target-cluster cells with both gene expression and peak accessibility nonzero: P(expr != 0 AND acc != 0) in the cluster.
- FC.expr_given_acc = expr_not_0.given_acc / expr_not_0: Fold-enrichment of gene expression among accessible cells relative to the cluster baseline expression rate: So values above 1 mean expression is enriched among cells where the peak is open

#### .bg: Background: same peak, other clusters: 
These are computed on the same peak and same gene, but using all cells outside the target cluster.
.all: same peak, All cells:
These are the same metrics as above but computed using all cells for the same gene and same peak.

#### Per-seed peak counts
- 'promotersLinkedToSeed': Number of linked peaks for this (gene, cluster) whose regulatorType == "proximal"
- 'distalPeaksLinkedToSeed': Number of linked peaks for this (gene, cluster) whose regulatorType == "distal"

#### Convenience / derived filter columns
- 'acc_cells_cluster': Estimated number of accessible cells in the target cluster for this peak: acc_cells_cluster = acc_not_0 * cluster_size where cluster_size = table(Idents(object))[cluster]
- 'delta_expr_given_acc.same_peak_bg': Difference in conditional expression between target cluster and background, for the same peak: expr_not_0.given_acc - expr_not_0.given_acc.bg
(- 'delta_expr_given_acc.other_peaks.same_cluster': Difference between the target peak and the “other linked peaks of same gene” summary, inside the target cluster:
expr_not_0.given_acc - expr_not_0.given_acc.bg_other_peaks.same_cluster
'delta_expr_given_acc.other_peaks.all': Difference between the target peak’s across-all-cells conditional expression and the “other linked peaks of same gene” across-all-cells summary: expr_not_0.given_acc.all - expr_not_0.given_acc.bg_other_peaks.all)

#### Final filter flags
- 'pass_cluster_specific'
    - promoter requirement, if enabled: promotersLinkedToSeed > 0
    - p.value-t.test_highly.acc_BH < th
    - t-stat_highly.acc > cluster_t_min
    - acc_cells_cluster >= min.cells
    - expr_and_acc_not_0 >= cluster_expr_given_acc_min
    - expr_not_0.given_acc > expr_given_acc_th
    - FC.expr_given_acc > cluster_fc_min
    - delta_expr_given_acc.same_peak_bg >= cluster_delta_same_peak_bg_min
- 'pass_global'
    - promoter requirement, if enabled
    - signac.zscores >= global_signac_z_min
    - signac.pvalues < global_signac_p_cutoff
    - expr_and_acc_not_0.all >= global_expr_given_acc_min
    - expr_not_0.given_acc.all > expr_given_acc_th
    - FC.expr_given_acc.all > global_fc_min
- 'pass_any': TRUE if pass_cluster_specific | pass_global
- 'pass_type'

In [45]:
summary_col_labels_short <- c(
  gene = "Gene",
  cluster = "Cluster",
  annotation = "Annotation",
  peak = "Peak",
  regulatorType = "Class",
  signac.scores = "Link Score",
  signac.zscores = "Link Z",
  signac.pvalues = "Link P",
  `t-stat_highly.acc` = "Acc. T-stat",
  `p.value-t.test_highly.acc_BH` = "Acc. FDR",
  acc_cells_cluster = "Accessible Cells",
  `expr_not_0.given_acc` = "P(expr|acc), cluster",
  `expr_not_0.given_acc.bg` = "P(expr|acc), bg",
  expr_and_acc_not_0 = "P(expr & acc), cluster",
  expr_and_acc_not_0.all = "P(expr & acc), all",
  FC.expr_given_acc = "Enrichment, cluster",
  FC.expr_given_acc.all = "Enrichment, all",
  delta_expr_given_acc.same_peak_bg = "Delta P(expr|acc)",
  promotersLinkedToSeed = "Promoter Peaks",
  distalPeaksLinkedToSeed = "Distal Peaks",
  pass_type = "Pass Type"
)

In [47]:
# keep only the selected columns, in this order
summary_cols <- c(
  "gene",
  "cluster",
  "annotation",
  "peak",
  "regulatorType",
  "signac.scores",
  "signac.zscores",
  "signac.pvalues",
  "t-stat_highly.acc",
  "p.value-t.test_highly.acc_BH",
  "acc_cells_cluster",
  "expr_not_0.given_acc",
  "expr_not_0.given_acc.bg",
  "expr_and_acc_not_0",
  "expr_and_acc_not_0.all",
  "FC.expr_given_acc",
  "FC.expr_given_acc.all",
  "delta_expr_given_acc.same_peak_bg",
  "promotersLinkedToSeed",
  "distalPeaksLinkedToSeed",
  "pass_type"
)


# filter + rename
peak_stats_summary <- object_w_net@misc$peak_stats.filtered[, summary_cols]
colnames(peak_stats_summary) <- summary_col_labels_short[summary_cols]

peak_stats_summary

,Gene,Cluster,Annotation,Peak,Class,Link Score,Link Z,Link P,Acc. T-stat,Acc. FDR,⋯,"P(expr|acc), cluster","P(expr|acc), bg","P(expr & acc), cluster","P(expr & acc), all","Enrichment, cluster","Enrichment, all",Delta P(expr|acc),Promoter Peaks,Distal Peaks,Pass Type
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr[1d]>
1,MYLK,0,module_genes,chr3-123766211-123767342,distal,0.09363708,3.276845,5.248694e-04,6.141285,2.363083e-09,⋯,0.6861314,0.3614458,0.06188282,0.02950192,1.018801,1.775816,0.3246856,3,9,cluster_specific
7,MYLK,0,module_genes,chr3-123815972-123817188,distal,0.06480712,3.966681,3.644017e-05,8.566264,7.019418e-17,⋯,0.7169811,0.3384615,0.07504937,0.03026820,1.064608,1.910201,0.3785196,3,9,cluster_specific
12,MYLK,0,module_genes,chr3-123883475-123884734,proximal,0.05038281,3.653168,1.295125e-04,8.727091,2.051297e-17,⋯,0.6808511,0.2511848,0.18959842,0.08563218,1.010961,1.478984,0.4296662,3,9,cluster_specific
99,KCNMA1,0,module_genes,chr10-77636810-77638795,proximal,0.05492740,4.224431,1.197725e-05,8.838366,9.343907e-18,⋯,0.6368564,0.1849711,0.15470704,0.06340996,1.073679,1.528471,0.4518853,2,6,cluster_specific
122,MYH11,0,module_genes,chr16-15856768-15857482,proximal,0.06555315,4.334615,7.300771e-06,5.760547,1.868771e-08,⋯,0.6875000,0.2439024,0.05069124,0.02049808,1.112154,1.787042,0.4435976,6,6,cluster_specific
123,MYH11,0,module_genes,chr16-15857870-15858601,proximal,0.05493147,3.406572,3.289211e-04,7.703362,6.008124e-14,⋯,0.7116564,0.2654321,0.07636603,0.03045977,1.151231,1.920139,0.4462243,6,6,cluster_specific
98,KCNMA1,0,seed_genes,chr10-77636810-77638795,proximal,0.07448205,2.930289,1.693235e-03,8.838366,9.343907e-18,⋯,0.6368564,0.1849711,0.15470704,0.06340996,1.073679,1.528471,0.4518853,2,6,cluster_specific
120,MYH11,0,seed_genes,chr16-15856768-15857482,proximal,0.09424665,1.802430,3.573887e-02,5.760547,1.868771e-08,⋯,0.6875000,0.2439024,0.05069124,0.02049808,1.112154,1.787042,0.4435976,6,6,cluster_specific
125,MYH11,0,seed_genes,chr16-15857870-15858601,proximal,0.07845987,1.726382,4.213940e-02,7.703362,6.008124e-14,⋯,0.7116564,0.2654321,0.07636603,0.03045977,1.151231,1.920139,0.4462243,6,6,cluster_specific


In [48]:
write.csv(peak_stats_summary, "exmpl_results_breast/peak_stats_summary_smooth_muscle_contraction.csv", row.names = FALSE)

## motif stats: Motif enrichment and footprint scoring
#### Identity Columns
- cluster
- gene
- motif
- TF: Transcription factor that binds motif (if multiple, new row for each)

#### Motif enrichment in linked peaks: motif_enrichment_per_gene() / calculate_enrichments()
logic: 
1. find peaks linked to the gene in the chosen cluster
2. split them into proximal vs distal using regulatorType
3. count how often each motif occurs in those linked peaks
4. compare that to a random background made by replacing each linked peak with a sampled “comparable”(GC content and length) peak from the same peak meta-feature cluster. t-test for enrichment
-> for this: make clusters of comparable peaks, store in object$peaks@meta.features$cluster
##### Proximal motif columns
- proximal.motif_count: The number of occurrences of this motif in proximal peaks linked to the gene in this cluster
- proximal.background_count: The mean motif count in matched random background peak sets for the proximal linked peaks. For each background draw, every linked proximal peak is replaced by a sampled peak from the same object$peaks@meta.features$cluster, then motif counts are computed, and the mean over draws is stored here
- log2FC.proximal: log2(proximal.motif_count / proximal.background_count): log2 enrichment of proximal motif count over matched background. Positive values mean enrichment; zero means no enrichment; negative values mean depletion
- t_stat.proximal: The t-statistic from testing the proximal background distribution against the foreground proximal count using
t.test(col_bg, mu = value_fg, alternative = "less") in effect. Interpreted practically, small p-values support that the background mean is lower than the foreground count, i.e. the motif is enriched in the linked proximal peaks.
- p_value.proximal: The raw p-value for that proximal enrichment test.
- p_adjust.proximal: A simple multiplicity-adjusted proximal p-value, computed as raw p_value.proximal * number_of_motifs, then capped at 1. This is Bonferroni-like, not BH/FDR.

##### Distal motif columns: analog to proximal, but using distal peaks
- distal.motif_count
- distal.background_count
- log2FC.distal = log2(distal.motif_count / distal.background_count)
- t_stat.distal
- p_value.distal
- p_adjust.distal

##### Promoter motif column
- promoter.motif_count: The number of occurrences of this motif in the gene’s promoter sequence, regardless of whether there is an accessible linked peak there. This is computed by get_tf_bindingsites_in_region(), which scans the promoter DNA sequence with the motif PWM. The promoter length: 2000 bp upstream TSS, but it may be extended if the gene has proximal linked peaks farther upstream than 2 kb.

#### Footprint columns: add_motif_stats() -> footprint_stats_test() for each  (gene, cluster, motif)
idea: The foreground footprint is built from motif sites found in peaks linked to the gene; the background footprint distribution is built by repeatedly sampling the same number of motif sites of that motif from elsewhere and rescoring them
core = mean.footprint - mean.flanks
- difference = observed.insertions_normalized - expected.insertions 
- mean.flanks = mean(difference in left + right flank windows)
- mean.footprint = mean(difference in motif-core window)
more negative footprint_score means the motif center is more depleted relative to the flanks, which is the classic “footprint” pattern
values near 0 mean little difference between core and flanks
positive values would mean the core is more accessible than the flanks

observed.insertions_normalized: empirical insertion profile around the motif sites, normalized by its global mean (it is the mean observed Tn5 insertion signal at each relative position, normalized so the average across positions is 1)
expected.insertions: sequence-bias-based expected insertion profile, normalized by flank expectation  (comes from GetExpectedInsertion(), which extracts the DNA sequence around the motif regions, gets the assay’s Tn5 bias vector, and calls FindExpectedInsertions(). That helper computes the expected insertion profile from local sequence composition and Tn5 hexamer bias, then normalizes that expected profile by the mean of the flank positions.)

##### Footprint statistic columns
- footprint_scores: as explained above
- bg_size: how many bg scores where computed
- bg_footprint_mean: The mean of the background footprint scores across those sampled background footprints
- footprint.t_stat: t.test(x = background_scores, mu = foreground_score, alternative = "greater"): So the test is asking whether the background mean score is greater than the foreground score. Because more negative scores are stronger footprints in your setup, a small p-value supports the foreground having a stronger depletion footprint than background
- footprint.p_value: The raw p-value for that footprint-vs-background test.
- footprint.p_value_adj: A per-(gene, cluster) multiplicity-adjusted version of footprint.p_value. In add_motif_stats(), after all motif rows are combined, the code multiplies each raw footprint p-value by the number of motif rows for that same gene and cluster, then caps at 1. So again this is Bonferroni-like, not BH/FDR.

##### Footprint quality / coverage columns
- sd.flanks: The standard deviation of the flank difference values for the foreground footprint. This is a variability measure for the left/right flank regions around the motif
- bg_sd_mean: The mean of the flank standard deviations across the sampled background footprints
- left_flank_nonzero_positions: How many positions in the left flank had nonzero observed insertion counts in the foreground footprint pileup. It is computed from observed.insertions.counts_sum in the footprint plot data
- right_flank_nonzero_positions: same for right flank

In [56]:

motif2TF_df <- data.frame(
  motif = rep(names(object_w_net@misc$motif2TF),
              lengths(object_w_net@misc$motif2TF)),
  TF = unlist(object_w_net@misc$motif2TF, use.names = FALSE),
  stringsAsFactors = FALSE
)

head(motif2TF_df)

,motif,TF
,<chr>,<chr>
1,MA0069.1,PAX6
2,MA0071.1,RORA
3,MA0074.1,RXRA
4,MA0074.1,VDR
5,MA0101.1,REL
6,MA0107.1,RELA


In [57]:

tmp <- object_w_net@misc$motif_stats %>%
  left_join(motif2TF_df, by = "motif")

Warning message in left_join(., motif2TF_df, by = "motif"):
"Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 89 of `x` matches multiple rows in `y`.
ℹ Row 5 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning."


In [52]:
motif_summary_cols <- c(
  "gene",
  "cluster",
  "TF",
  "motif",
  "proximal.motif_count",
  "proximal.background_count",
  "log2FC.proximal",
  "p_adjust.proximal",
  "distal.motif_count",
  "distal.background_count",
  "log2FC.distal",
  "p_adjust.distal",
  "promoter.motif_count",
  "footprint_score",
  "bg_footprint_mean",
  "footprint.p_value_adj",
  "bg_size",
  "sd.flanks",
  "bg_sd_mean",
  "left_flank_nonzero_positions",
  "right_flank_nonzero_positions"
)

In [60]:
motif_summary_col_labels <- c(
  gene = "Gene",
  cluster = "Cluster",
  TF = "TF",
  motif = "Motif",
  proximal.motif_count = "Prox Motif count",
  proximal.background_count = "Prox Bg count",
  log2FC.proximal = "Prox Log2FC",
  p_adjust.proximal = "Prox p-value adj",
  distal.motif_count = "Dist Motif count",
  distal.background_count = "Dist Bg count",
  log2FC.distal = "Dist Log2FC",
  p_adjust.distal = "Dist p-value adj",
  promoter.motif_count = "Prom Motif count",
  footprint_score = "FP Score",
  bg_footprint_mean = "Bg FP Score",
  footprint.p_value_adj = "FP p-value adj",
  bg_size = "Bg Size",
  sd.flanks = "Flank sd",
  bg_sd_mean = "Bg Flank sd",
  left_flank_nonzero_positions = "Left Flank != 0",
  right_flank_nonzero_positions = "Right Flank != 0"
)

In [58]:
colnames(tmp)

[1] "gene"                          "cluster"                      
 [3] "proximal.motif_count"          "proximal.background_count"    
 [5] "log2FC.proximal"               "t_stat.proximal"              
 [7] "p_value.proximal"              "p_adjust.proximal"            
 [9] "distal.motif_count"            "distal.background_count"      
[11] "log2FC.distal"                 "t_stat.distal"                
[13] "p_value.distal"                "p_adjust.distal"              
[15] "motif"                         "promoter.motif_count"         
[17] "footprint_score"               "bg_size"                      
[19] "bg_footprint_mean"             "footprint.t_stat"             
[21] "footprint.p_value"             "footprint.p_value_adj"        
[23] "sd.flanks"                     "bg_sd_mean"                   
[25] "left_flank_nonzero_positions"  "right_flank_nonzero_positions"
[27] "TF"

In [61]:
motif_stats_summary <- tmp[, motif_summary_cols]
colnames(motif_stats_summary) <- motif_summary_col_labels[motif_summary_cols]

motif_stats_summary

Gene,Cluster,TF,Motif,Prox Motif count,Prox Bg count,Prox Log2FC,Prox p-value adj,Dist Motif count,Dist Bg count,⋯,Dist p-value adj,Prom Motif count,FP Score,Bg FP Score,FP p-value adj,Bg Size,Flank sd,Bg Flank sd,Left Flank != 0,Right Flank != 0
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
KCNMA1,0,REL,MA0101.1,1,0.84,0.25,1,0,0,⋯,0,1,-0.34150,0.04059,1.000,20,1.939,3.354,3,3
KCNMA1,0,REL,MA0101.1,1,0.84,0.25,1,0,0,⋯,0,1,-0.34150,-0.49050,1.000,20,1.939,3.927,3,3
KCNMA1,0,RELA,MA0107.1,1,1,0,1,0,0,⋯,0,2,-0.34150,-0.40420,1.000,20,1.939,3.314,3,3
KCNMA1,0,RELA,MA0107.1,1,1,0,1,0,0,⋯,0,2,-0.34150,-0.31480,1.000,20,1.939,3.670,3,3
KCNMA1,0,EWSR1-FLI1,MA0149.1,10,6.5,0.62,0.00062,0,0,⋯,0,0,0.02615,0.07307,1.000,20,1.019,1.194,17,20
KCNMA1,0,EWSR1-FLI1,MA0149.1,10,6.5,0.62,0.00062,0,0,⋯,0,0,0.02615,0.29410,0.530,20,1.019,1.023,17,20
KCNMA1,0,INSM1,MA0155.1,3,2.6,0.21,1,0,0,⋯,0,1,0.34600,-0.20790,1.000,20,1.265,2.255,9,11
KCNMA1,0,INSM1,MA0155.1,3,2.6,0.21,1,0,0,⋯,0,1,0.34600,-0.10530,1.000,20,1.265,2.090,9,11
KCNMA1,0,PLAG1,MA0163.1,1,4.8,-2.3,1,0,0,⋯,0,0,-0.06973,0.17060,1.000,20,2.218,3.250,4,7


In [62]:
write.csv(motif_stats_summary, "exmpl_results_breast/motif_stats_summary_smooth_muscle_contraction.csv", row.names = FALSE)

## network

In [26]:
head(object_w_net@misc$context_subNetwork$smooth_muscle_contraction_cluster_0_genes_with_peaks)

,from,to,color,priorTF,reg_type,in.prom
,<chr>,<chr>,<chr>,<lgl>,<int>,<int>
KCNMA1.1,REL,KCNMA1,red,FALSE,1,3
KCNMA1.2,RELA,KCNMA1,red,FALSE,1,3
KCNMA1.3,PLAG1,KCNMA1,red,FALSE,1,1
KCNMA1.4,FOXG1,KCNMA1,yellow,FALSE,1,1
KCNMA1.5,MSC,KCNMA1,yellow,FALSE,1,3
KCNMA1.6,ZIC1,KCNMA1,yellow,FALSE,1,3


In [27]:
library(dplyr)
library(jsonlite)



Attaching package: 'jsonlite'


The following object is masked from 'package:purrr':

    flatten




In [1]:
library(dplyr)
library(jsonlite)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [29]:

# assumed columns: from, to, color, priorTF, reg_type, in.prom
df <- object_w_net@misc$context_subNetwork$smooth_muscle_contraction_cluster_0_genes_with_peaks
links <- df %>%
  transmute(
    source = from,
    target = to,
    edge_color = color,
    edge_width = reg_type,
    edge_dash = ifelse(in.prom == 1, "4,4", "0")
  )

# node table
from_nodes <- df %>%
  distinct(name = from, priorTF) %>%
  mutate(type = "from")

to_nodes <- df %>%
  distinct(name = to) %>%
  mutate(priorTF = NA, type = "to")

nodes <- bind_rows(from_nodes, to_nodes) %>%
  group_by(name) %>%
  summarise(
    priorTF = dplyr::first(na.omit(priorTF)),
    type = ifelse(any(type == "from"), "from", "to"),
    .groups = "drop"
  ) %>%
  mutate(
    node_fill = case_when(
      type == "to" ~ "#cccccc",          # default for target nodes
      priorTF %in% TRUE ~ "#ff7f0e",     # prior TF
      priorTF %in% FALSE ~ "#1f77b4",    # not prior TF
      TRUE ~ "#999999"
    )
  )

# convert source/target names to node indices for D3
nodes <- nodes %>% mutate(id = row_number() - 1)

links <- links %>%
  left_join(nodes %>% select(source = name, source_id = id), by = "source") %>%
  left_join(nodes %>% select(target = name, target_id = id), by = "target") %>%
  transmute(
    source = source_id,
    target = target_id,
    edge_color,
    edge_width,
    edge_dash
  )

graph <- list(
  nodes = nodes,
  links = links
)


In [31]:
graph

name,priorTF,type,node_fill,id
<chr>,<lgl>,<chr>,<chr>,<dbl>
ASCL1,FALSE,from,#1f77b4,0
BCL6B,FALSE,from,#1f77b4,1
BHLHE22,FALSE,from,#1f77b4,2
BHLHE41,FALSE,from,#1f77b4,3
CREB3L1,FALSE,from,#1f77b4,4
CREB3L4,FALSE,from,#1f77b4,5
CTCF,FALSE,from,#1f77b4,6
DUX4,FALSE,from,#1f77b4,7
E2F8,FALSE,from,#1f77b4,8


In [32]:

write_json(graph, "exmpl_results_breast/graph.json", auto_unbox = TRUE, pretty = TRUE)